# WomenInAmericanReadingCulture

This notebook explores the visbility of women in the **Goodreads Top Ranked Novels** dataset and the **New York Times Hardcover Fiction Bestsellers** dataset comparative to men. In this analysis, I particularly focus on American authors and American female authors as it connects to my broader theme of a women's influence historically in the United States in culture and politics. 

The main questions that I aim to explore in this anlysis are as follows:
### Main Questions
1. How visible are women authors in the Goodreads Top Ranked Novels dataset overall (compared to men)?
2. What changes appear when I focus only on American authors (compared to men)?
3. How does women's representation shift across publication decades?
4. Which American women-authored books appear in both the Goodreads Top Ranked Novels dataset and the **New York Times Hardcover Fiction Bestsellers** dataset (compared to men)?

The goal is to understand if and how female authors were included/represented in places of prestige in the literary landscape where publications share ideas, perspectives, and sentiments as a means to exert cultural influence.

In [9]:
import pandas as pd
import numpy as np
import altair as alt
import re
from IPython.display import display
from pathlib import Path

alt.data_transformers.disable_max_rows()

print("Current folder:", Path.cwd())
print("Files here:")
for f in Path.cwd().iterdir():
    print(f.name)

top_df = pd.read_csv("https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv", sep="\t", encoding='utf-8')
nyt_df = pd.read_csv("https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv", sep="\t", encoding='utf-8')

print("Top 500 novels shape:", top_df.shape)
print("NYT titles shape:", nyt_df.shape)

display(top_df.head())
display(nyt_df.head())

Current folder: /Users/ananyakavatekar/Desktop/is310 Apr 23
Files here:
.DS_Store
WomenInAmericanReadingCulture_FINAL.ipynb
WomenAndPublicVoiceInAmericanReadingCulture.ipynb
WomenInAmericanReadingCulture_updated.ipynb
WomenInAmericanReadingCulture_FINAL_fixed_renderer.ipynb
Top 500 novels shape: (500, 29)
NYT titles shape: (60386, 6)


,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_num_ratings,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,"269,435","12,053",318,211,1.810748e+09,17220427,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,"561,016","15,380",172,133,1.156132e+10,66462036,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,"1,262,480","19,440",373,68,3.373178e+09,50566653,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,"931,898","13,603",301,88,3.373178e+09,50566653,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt,NaN
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,"486,155","16,307",368,145,3.434000e+03,95207986,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...,NaN


,year,week,rank,title_id,title,author
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck
3,1931,1931-10-12,4,4038,SHADOWS ON THE ROCK,Willa Cather
4,1931,1931-10-12,5,3946,SCARMOUCHE THE KING MAKER,Rafael Sabatini


In [10]:
alt.renderers.enable("html")

RendererRegistry.enable('html')

## Cleaning and setup

In [11]:
# Centralized cleaning so the visualization cells can stay short

top_books = top_df.copy()
top_books["author_gender"] = top_books["author_gender"].astype(str).str.strip().str.lower()
top_books["author_nationality"] = top_books["author_nationality"].fillna("").astype(str)
top_books["pub_year"] = pd.to_numeric(top_books["pub_year"], errors="coerce")
top_books["top_500_rank"] = pd.to_numeric(top_books["top_500_rank"], errors="coerce")
top_books["gr_num_ratings"] = pd.to_numeric(top_books["gr_num_ratings"], errors="coerce")
top_books["gr_num_ratings_rank"] = pd.to_numeric(top_books["gr_num_ratings_rank"], errors="coerce")

us_pattern = r"american|united states|u\.s\.|usa|us"
top_books["is_us_author"] = top_books["author_nationality"].str.lower().str.contains(us_pattern, regex=True)

top_books["pub_decade"] = (np.floor(top_books["pub_year"] / 10) * 10).astype("Int64")

# Minimal matching keys for dataset overlap
top_books["title_key"] = top_books["title"].astype(str).str.strip().str.lower()
top_books["author_key"] = top_books["author"].astype(str).str.strip().str.lower()

nyt_books = nyt_df.copy()
nyt_books["year"] = pd.to_numeric(nyt_books["year"], errors="coerce")
nyt_books["rank"] = pd.to_numeric(nyt_books["rank"], errors="coerce")
nyt_books["title_key"] = nyt_books["title"].astype(str).str.strip().str.lower()
nyt_books["author_key"] = nyt_books["author"].astype(str).str.strip().str.lower()

top_us = (
    top_books[
        top_books["is_us_author"] &
        top_books["author_gender"].isin(["female", "male"])
    ]
    .copy()
)

top_us_unique = top_us.drop_duplicates(subset=["title_key", "author_key"]).copy()

nyt_summary = (
    nyt_books
    .groupby(["title_key", "author_key"], as_index=False)
    .agg(
        nyt_title=("title", "first"),
        nyt_author=("author", "first"),
        first_year=("year", "min"),
        best_rank=("rank", "min"),
        weeks_on_list=("week", "nunique")
    )
)

overlap_us = (
    top_us_unique
    .merge(nyt_summary, on=["title_key", "author_key"], how="inner")
    .copy()
)

overlap_summary = (
    top_us_unique.groupby("author_gender", as_index=False)
    .size()
    .rename(columns={"size": "top500_count"})
    .merge(
        overlap_us.groupby("author_gender", as_index=False)
        .size()
        .rename(columns={"size": "overlap_count"}),
        on="author_gender",
        how="left"
    )
    .fillna({"overlap_count": 0})
)

overlap_summary["overlap_count"] = overlap_summary["overlap_count"].astype(int)
overlap_summary["overlap_rate"] = 100 * overlap_summary["overlap_count"] / overlap_summary["top500_count"]

overlap_count_long = pd.DataFrame({
    "author_gender": list(overlap_summary["author_gender"]) * 2,
    "group": ["US Top 500"] * len(overlap_summary) + ["US Top 500 + NYT overlap"] * len(overlap_summary),
    "count": list(overlap_summary["top500_count"]) + list(overlap_summary["overlap_count"])
})

overlap_women = (
    overlap_us[overlap_us["author_gender"] == "female"]
    .copy()
    .sort_values(["weeks_on_list", "best_rank", "top_500_rank"], ascending=[False, True, True])
)
overlap_women["title_author"] = overlap_women["title"] + " — " + overlap_women["author"]
overlap_women["weeks_label"] = overlap_women["weeks_on_list"].astype(int).astype(str)

female_decades = (
    top_books[
        top_books["author_gender"].isin(["female", "male"]) &
        top_books["pub_year"].notna() &
        (top_books["pub_year"] >= 1800)
    ]
    .groupby(["pub_decade", "author_gender"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

decade_totals = (
    female_decades.groupby("pub_decade", as_index=False)["count"]
    .sum()
    .rename(columns={"count": "decade_total"})
)

female_decades = female_decades.merge(decade_totals, on="pub_decade", how="left")
female_decades = female_decades[female_decades["decade_total"] >= 5].copy()
female_decades["pct"] = 100 * female_decades["count"] / female_decades["decade_total"]
female_decades = female_decades[female_decades["author_gender"] == "female"].copy()

color_scale = alt.Scale(
    domain=["female", "male"],
    range=["#C06C84", "#4C78A8"]
)

display(overlap_summary)

,author_gender,top500_count,overlap_count,overlap_rate
0,female,93,24,25.806452
1,male,164,70,42.682927


## 1. How visible are women authors in the Goodreads Top Ranked Novels dataset overall?

In [12]:
gender_counts = (
    top_books[top_books["author_gender"].isin(["female", "male"])]
    .groupby("author_gender", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

chart1 = (
    alt.Chart(gender_counts)
    .mark_bar(cornerRadiusTopLeft=8, cornerRadiusTopRight=8, width=160)
    .encode(
        x=alt.X("author_gender:N", title="Author gender", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("count:Q", title="Number of novels"),
        color=alt.Color("author_gender:N", scale=color_scale, legend=None),
        tooltip=[
            alt.Tooltip("author_gender:N", title="Gender"),
            alt.Tooltip("count:Q", title="Books")
        ]
    )
    .properties(
        title="Top 500 Novels: Count of Books by Author Gender",
        width=420,
        height=320
    )
)

labels1 = (
    alt.Chart(gender_counts)
    .mark_text(dy=-8, fontSize=13, fontWeight="bold")
    .encode(
        x=alt.X("author_gender:N"),
        y=alt.Y("count:Q"),
        text=alt.Text("count:Q")
    )
)

display((chart1 + labels1).configure_view(stroke=None))

alt.LayerChart(...)

### Analysis

This chart shows a clear gender imbalance in the Goodreads Top 500 novels dataset. Male authors dominate this prestigious space and account for most of the books in the list, while women take a much smaller share. 
This shows us that female authors do hold a strong presence in this space but not nearly on equal terms with men. This of course follows the predictable trend across history. On nuance here however is that this graph represents books 
in the Top 500 over unique authors so there may be a case where a few very popular male authors that have written a series of books imflate the disparity farther.

## 2. What changes appear when I focus only on American authors?

In [13]:
comparison_source = top_books[top_books["author_gender"].isin(["female", "male"])].copy()

overall_counts = (
    comparison_source
    .groupby("author_gender", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
overall_counts["group"] = "All Top 500"

us_counts = (
    comparison_source[comparison_source["is_us_author"]]
    .groupby("author_gender", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
us_counts["group"] = "American Only"

compare_counts = pd.concat([overall_counts, us_counts], ignore_index=True)

group_totals = (
    compare_counts
    .groupby("group", as_index=False)["count"]
    .sum()
    .rename(columns={"count": "group_total"})
)

compare_counts = compare_counts.merge(group_totals, on="group", how="left")
compare_counts["pct"] = 100 * compare_counts["count"] / compare_counts["group_total"]

gap_df = compare_counts.pivot(index="group", columns="author_gender", values="pct").reset_index()
gap_df["gap_pp"] = gap_df["male"] - gap_df["female"]
gap_df["label"] = gap_df["gap_pp"].round(1).astype(str) + " percentage points"

compare_chart = (
    alt.Chart(compare_counts)
    .mark_bar(cornerRadiusTopLeft=8, cornerRadiusTopRight=8, width=55)
    .encode(
        x=alt.X("group:N", title=None, axis=alt.Axis(labelAngle=0)),
        xOffset=alt.XOffset("author_gender:N"),
        y=alt.Y("pct:Q", title="Percent of books", scale=alt.Scale(domain=[0, 100])),
        color=alt.Color("author_gender:N", scale=color_scale, title="Author gender"),
        tooltip=[
            alt.Tooltip("group:N", title="Subset"),
            alt.Tooltip("author_gender:N", title="Gender"),
            alt.Tooltip("count:Q", title="Books"),
            alt.Tooltip("pct:Q", title="Percent", format=".1f")
        ]
    )
    .properties(
        title="Gender Representation: All Top 500 vs American Authors",
        width=430,
        height=320
    )
)

compare_labels = (
    alt.Chart(compare_counts)
    .mark_text(dy=-8, fontSize=12, fontWeight="bold")
    .encode(
        x=alt.X("group:N"),
        xOffset=alt.XOffset("author_gender:N"),
        y=alt.Y("pct:Q"),
        text=alt.Text("pct:Q", format=".1f")
    )
)

gap_chart = (
    alt.Chart(gap_df)
    .mark_bar(cornerRadiusEnd=8, size=45, color="#6C63FF")
    .encode(
        y=alt.Y("group:N", title=None),
        x=alt.X("gap_pp:Q", title="Male share − Female share (percentage points)"),
        tooltip=[
            alt.Tooltip("group:N", title="Subset"),
            alt.Tooltip("male:Q", title="Male %", format=".1f"),
            alt.Tooltip("female:Q", title="Female %", format=".1f"),
            alt.Tooltip("gap_pp:Q", title="Gap", format=".1f")
        ]
    )
    .properties(
        title="The Gap Shrinks When Focusing on American Authors",
        width=330,
        height=160
    )
)

gap_labels = (
    alt.Chart(gap_df)
    .mark_text(align="left", dx=6, fontSize=12, fontWeight="bold")
    .encode(
        y=alt.Y("group:N"),
        x=alt.X("gap_pp:Q"),
        text=alt.Text("label:N")
    )
)

final_comparison = alt.hconcat(
    compare_chart + compare_labels,
    gap_chart + gap_labels
).configure_view(stroke=None)

display(final_comparison)

alt.HConcatChart(...)

### Analysis
When we focus on only American authors, we see that the disparity remains. However, the gap in literary publications between male and female authors that shoot to the prestigious rankings reducded considerably. Female authors here still represent the minority but the gap from 42 percentage points in the overall dataset including all authors globally reduced to 27.6 percentage points. This suggests that female author visbility in the United States is somewhat stronger than female authors across the globe.

## 3. How does women's representation shift across publication decades?

In [14]:
female_chart = (
    alt.Chart(female_decades)
    .mark_line(point=True, strokeWidth=3, color="#C06C84")
    .encode(
        x=alt.X("pub_decade:Q", title="Publication decade"),
        y=alt.Y("pct:Q", title="Percent of books by women", scale=alt.Scale(domain=[0, 100])),
        tooltip=[
            alt.Tooltip("pub_decade:Q", title="Decade"),
            alt.Tooltip("count:Q", title="Women-authored books"),
            alt.Tooltip("decade_total:Q", title="Total books in decade"),
            alt.Tooltip("pct:Q", title="Female share", format=".1f")
        ]
    )
    .properties(
        title="Women's Representation Across Publication Decades",
        width=650,
        height=350
    )
)

display(female_chart.configure_view(stroke=None))

alt.Chart(...)

### Analysis
This graph showcases that female author representation is not be unchanging or following a linear upwards or downwards trend across history. The percentage of books authored by female writers changed darastically across publication years. This suggests that literary visbility has been and is incredibly varible raising questions as to what public sentiments, phenonmeas, political/social events make it so. The overall trend shows that there has been stronger female representation in recent periods (particularly since the 1950s) compared to prior time periods. However, this trend is incredibly choppy, many times is sharply rises and falls indicating possible phenomenas that resulted in reduced female author visbility. One intresting pattern was the peak in visbility seen during the 1800 time period and and sharp gradual decline since then. 

## 4. Which American women-authored books appear in both the Goodreads Top Ranked Novels dataset and the New York Times Hardcover Fiction Bestsellers dataset?

In [16]:
count_chart = (
    alt.Chart(overlap_count_long)
    .mark_bar(cornerRadiusTopLeft=8, cornerRadiusTopRight=8, width=55)
    .encode(
        x=alt.X("group:N", title=None, axis=alt.Axis(labelAngle=0)),
        xOffset=alt.XOffset("author_gender:N"),
        y=alt.Y("count:Q", title="Number of books"),
        color=alt.Color("author_gender:N", scale=color_scale, title="Author gender"),
        tooltip=[
            alt.Tooltip("group:N", title="Group"),
            alt.Tooltip("author_gender:N", title="Gender"),
            alt.Tooltip("count:Q", title="Books")
        ]
    )
    .properties(
        title="U.S. Top 500 Books and Their NYT Overlap, by Author Gender",
        width=430,
        height=320
    )
)

count_labels = (
    alt.Chart(overlap_count_long)
    .mark_text(dy=-8, fontSize=12, fontWeight="bold")
    .encode(
        x=alt.X("group:N"),
        xOffset=alt.XOffset("author_gender:N"),
        y=alt.Y("count:Q"),
        text=alt.Text("count:Q")
    )
)

rate_chart = (
    alt.Chart(overlap_summary)
    .mark_bar(cornerRadiusTopLeft=8, cornerRadiusTopRight=8, width=120)
    .encode(
        x=alt.X("author_gender:N", title="Author gender", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("overlap_rate:Q", title="Percent of U.S. Top 500 books also in NYT"),
        color=alt.Color("author_gender:N", scale=color_scale, legend=None),
        tooltip=[
            alt.Tooltip("author_gender:N", title="Gender"),
            alt.Tooltip("top500_count:Q", title="U.S. Top 500 books"),
            alt.Tooltip("overlap_count:Q", title="Overlap books"),
            alt.Tooltip("overlap_rate:Q", title="Overlap rate", format=".1f")
        ]
    )
    .properties(
        title="How Often U.S. Top 500 Books Also Appear in NYT, by Author Gender",
        width=300,
        height=320
    )
)

rate_labels = (
    alt.Chart(overlap_summary)
    .mark_text(dy=-8, fontSize=12, fontWeight="bold")
    .encode(
        x=alt.X("author_gender:N"),
        y=alt.Y("overlap_rate:Q"),
        text=alt.Text("overlap_rate:Q", format=".1f")
    )
)

display(
    alt.hconcat(count_chart + count_labels, rate_chart + rate_labels)
    .configure_view(stroke=None)
)

alt.HConcatChart(...)

### Analysis
This comparison seperates the visbility seen in the two rankings (GoodReads and NYT Fiction Bestsellers). Male authors still make up a majority of the overlap where their publications made both prestigeous lists, however this analysis is still interesting as it represents broader visbility and appeal. The general auidence in both of these prestigeous lists can be somewhat different in nature where GoodReads incompasses many online/digital readers whereas NYT incompasses a broader public that purchases physical ficition books. This analysis raises furhter questions in the qualities that skew towards resulting in the publication overlapping between the two spaces of visibility. 

In [17]:
books_chart = (
    alt.Chart(overlap_women)
    .mark_bar(cornerRadiusEnd=8, color="#C06C84")
    .encode(
        y=alt.Y(
            "title_author:N",
            sort=alt.SortField(field="weeks_on_list", order="descending"),
            title=None
        ),
        x=alt.X("weeks_on_list:Q", title="Weeks on New York Times bestseller list"),
        tooltip=[
            alt.Tooltip("title:N", title="Title"),
            alt.Tooltip("author:N", title="Author"),
            alt.Tooltip("top_500_rank:Q", title="Top 500 rank"),
            alt.Tooltip("pub_year:Q", title="Publication year"),
            alt.Tooltip("best_rank:Q", title="Best NYT rank"),
            alt.Tooltip("weeks_on_list:Q", title="Weeks on NYT list"),
            alt.Tooltip("first_year:Q", title="First NYT year")
        ]
    )
    .properties(
        title="American Women-Authored Books Appearing in Both the Top 500 and NYT Bestseller Dataset",
        width=650,
        height=alt.Step(28)
    )
)

books_labels = (
    alt.Chart(overlap_women)
    .mark_text(
        align="left",
        baseline="middle",
        dx=6,
        fontSize=11,
        fontWeight="bold",
        color="#C06C84"
    )
    .encode(
        y=alt.Y(
            "title_author:N",
            sort=alt.SortField(field="weeks_on_list", order="descending")
        ),
        x=alt.X("weeks_on_list:Q"),
        text=alt.Text("weeks_label:N")
    )
)

display((books_chart + books_labels).configure_view(stroke=None))

display(
    overlap_women[
        ["title", "author", "top_500_rank", "pub_year", "best_rank", "weeks_on_list", "first_year"]
    ].reset_index(drop=True)
)

alt.LayerChart(...)

,title,author,top_500_rank,pub_year,best_rank,weeks_on_list,first_year
0,The Help,Kathryn Stockett,206,2009,1,108,2009
1,To Kill a Mockingbird,Harper Lee,29,1960,2,98,1960
2,Gone Girl,Gillian Flynn,324,2012,1,80,2012
3,Gone with the Wind,Margaret Mitchell,77,1936,1,78,1936
4,The Lovely Bones,Alice Sebold,208,2002,1,65,2002
5,A Tree Grows in Brooklyn,Betty Smith,187,1943,1,51,1943
6,The Mammoth Hunters,Jean M. Auel,434,1985,1,47,1985
7,The Yearling,Marjorie Kinnan Rawlings,164,1938,1,42,1938
8,The Joy Luck Club,Amy Tan,236,1989,3,35,1989
9,Go Set a Watchman,Harper Lee,481,2015,1,33,2015


### Analysis

This final chart adds greater insight into the overalap of female authors on both lists. This chart is ranked from most to least by the weeks that their work appeared on the NYT list which adds an added dimension to the level of popularity and visbility that the work and author recieved to those that follow the bestseller list. All together, these titles show that women did recieve both mainstream success but also were dampered compared to male authors even in a highly selective overlap.

## Conclusion

Overall, these visualizations show that female authors are visible in American (and broader Global) reading culture, but they are not on equal footing with male authors. In the Goodreads Top 500 dataset female authors are clearly underrepresented and the imbalance remains even when the anlysis is narrowed to only American authors. The popular publications across decades suggests that over time there were darastic shifts in female author visibility while the overlap with the NYT bestsellers list showcase that there are some traits that contributed to some female authors recieving widerspread long term recognition and bestseller success in a broader reading culture. All together, that analysis shows that literary visbility and acheivement remains extremely uneven between male and female authors across the globe with some narrowing of the disparity in subsections such as when considering only American authors which may extend to other western landscapes.
